# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Build the feature vector.

I use five numeric features measured from the early window or available by March 20.

I apply log transformations to reduce skew in traffic-related values. Missing numeric values are handled with median imputation inside the model pipeline.

No categorical columns are selected in this version, so no categorical encoding is applied. Client and content IDs are kept only for grouping and joining, never as model features.


In [34]:
import duckdb
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download
from google.colab import userdata

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

print("Libraries imported successfully.")

Libraries imported successfully.



## 2. Feature notes

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| impressions_log | Early-window impressions | Median imputation | Yes, March 1–20 |
| avg_position_early | Average search position | Median imputation | Yes, March 1–20 |
| days_since_update | Days since last content update | Median imputation | Yes, as of March 20 |
| search_volume_log | Log of keyword search volume | Median imputation | Timing needs confirmation |
| impr_trend_log | Early impressions trend | Median imputation | Yes, March 1–20 |

All selected features are numeric. No categorical feature is used, so categorical encoding is not needed.

The exact measurement timestamp for search volume is not available in this query. I treat it as provisional until its timestamp is confirmed.


In [35]:
TOKEN = userdata.get("HF_TOKEN")

if not TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

REPO = "FlyRank/internship-warehouse"

FACT = hf_hub_download(
    REPO,
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=TOKEN
)

DIM_CONTENT = hf_hub_download(
    REPO,
    "dim_content.parquet",
    repo_type="dataset",
    token=TOKEN
)

con = duckdb.connect()

print("Warehouse files loaded successfully.")

Warehouse files loaded successfully.



## 3. The leakage hunt

The label uses clicks from March 21–31, so future clicks must never be model features.

`ctr_early` shares early clicks with the label formula. This is a target-construction dependency, although it does not directly contain future data. I test the model with and without it.

I also test an intentionally leaked feature (`label` itself). It should produce a perfect AUC and demonstrates why target-derived information must be excluded.

No existing product score or decision flag is included in the selected feature vector.


In [36]:
df = con.sql(f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_early,
        SUM(gsc_clicks) AS clicks_early,
        AVG(gsc_avg_position) AS avg_position_early,

        SUM(gsc_impressions) FILTER (
            WHERE report_date <= DATE '2026-03-10'
        ) AS imp_h1,

        SUM(gsc_impressions) FILTER (
            WHERE report_date > DATE '2026-03-10'
        ) AS imp_h2

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
    GROUP BY 1, 2
),

late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_late

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
    GROUP BY 1, 2
)

SELECT
    e.client_hash_id,
    e.content_hash_id,
    e.impressions_early,
    e.clicks_early * 1.0 /
        NULLIF(e.impressions_early, 0) AS ctr_early,
    e.avg_position_early,

    (COALESCE(e.imp_h2, 0) + 1.0) /
        (COALESCE(e.imp_h1, 0) + 1.0) AS impr_trend_early,

    DATE '2026-03-20' - c.content_updated_date
        AS days_since_update,

    c.search_volume,
    COALESCE(l.clicks_late, 0) AS clicks_late,

    (
        COALESCE(l.clicks_late, 0) / 11.0
        < 0.8 * (e.clicks_early / 20.0)
    ) AS label

FROM early e
JOIN read_parquet('{DIM_CONTENT}') c
    USING (client_hash_id, content_hash_id)
LEFT JOIN late l
    USING (client_hash_id, content_hash_id)

WHERE e.clicks_early >= 5
  AND c.content_updated_date <= DATE '2026-03-20'
""").df()

df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
df["search_volume"] = df["search_volume"].astype(float)

print("Dataset shape:", df.shape)
print("Positive label rate:", round(df["label"].mean(), 3))

Dataset shape: (2526, 10)
Positive label rate: 0.59


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [37]:
df["impressions_log"] = np.log1p(df["impressions_early"])

df["search_volume_log"] = np.log1p(
    df["search_volume"].clip(lower=0)
)

df["impr_trend_log"] = np.log(
    df["impr_trend_early"].clip(0.05, 20)
)

NUMERIC_FEATURES = [
    "impressions_log",
    "avg_position_early",
    "days_since_update",
    "search_volume_log",
    "impr_trend_log"
]

FEATS = NUMERIC_FEATURES

CATEGORICAL_FEATURES = []

FEATS = NUMERIC_FEATURES
LABEL = "label"
GROUP = "client_hash_id"

d = df.dropna(subset=[LABEL, GROUP]).copy()

print("Feature vector shape:", d[FEATS].shape)
print("Features:", FEATS)
print("Categorical features:", CATEGORICAL_FEATURES)
print("\nMissing values:")
print(d[FEATS].isna().sum())

Feature vector shape: (2526, 5)
Features: ['impressions_log', 'avg_position_early', 'days_since_update', 'search_volume_log', 'impr_trend_log']
Categorical features: []

Missing values:
impressions_log        0
avg_position_early     0
days_since_update      0
search_volume_log     40
impr_trend_log         0
dtype: int64


In [38]:
feature_notes = pd.DataFrame({
    "feature": FEATS,
    "dtype": [d[f].dtype for f in FEATS],
    "missing_count": [d[f].isna().sum() for f in FEATS],
    "missing_pct": [
        round(d[f].isna().mean() * 100, 2)
        for f in FEATS
    ]
})

display(feature_notes)
print("Categorical columns selected:", len(CATEGORICAL_FEATURES))

,feature,dtype,missing_count,missing_pct
0,impressions_log,float64,0,0.00
1,avg_position_early,float64,0,0.00
2,days_since_update,float64,0,0.00
3,search_volume_log,float64,40,1.58
4,impr_trend_log,float64,0,0.00


Categorical columns selected: 0


In [39]:
# Check that forbidden columns are not model features
forbidden = [
    "label",
    "clicks_late",
    "client_hash_id",
    "content_hash_id"
]

print("Forbidden features found:",
      [f for f in forbidden if f in FEATS])

assert not any(f in FEATS for f in forbidden)

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=0
)

tr_i, te_i = next(
    gss.split(d, groups=d[GROUP])
)

train = d.iloc[tr_i].copy()
test = d.iloc[te_i].copy()

def evaluate_features(features):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=20,
            random_state=0,
            n_jobs=-1
        )
    )

    model.fit(train[features], train[LABEL])
    scores = model.predict_proba(test[features])[:, 1]

    return roc_auc_score(test[LABEL], scores)

features_with_ctr = FEATS + ["ctr_early"]
features_without_ctr = FEATS

auc_with_ctr = evaluate_features(features_with_ctr)
auc_without_ctr = evaluate_features(features_without_ctr)

print("AUC with ctr_early:", round(auc_with_ctr, 3))
print("AUC without ctr_early:", round(auc_without_ctr, 3))

# Intentional leakage demonstration — never use this feature
leaky_auc = roc_auc_score(
    test[LABEL],
    test[LABEL]
)

print("Intentional label leakage AUC:", round(leaky_auc, 3))

Forbidden features found: []
AUC with ctr_early: 0.597
AUC without ctr_early: 0.605
Intentional label leakage AUC: 1.0



## 4. What I excluded and why

| Excluded field | Reason |
|---|---|
| clicks_late | Comes from the future label window |
| label | Target variable, never an input |
| client_hash_id | Identifier; grouping only |
| content_hash_id | Identifier; joining only |
| Existing product scores / flags | Could encode an earlier decision |
| URLs, domains, private queries | Privacy and unnecessary identifiers |

The feature vector contains only the five selected numeric features.


In [40]:
excluded_fields = pd.DataFrame({
    "field": [
        "clicks_late",
        "label",
        "client_hash_id",
        "content_hash_id",
        "Existing product scores / flags",
        "URLs / domains / private queries"
    ],
    "reason": [
        "Future label window",
        "Target variable",
        "Identifier; grouping only",
        "Identifier; joining only",
        "Decision-derived information",
        "Privacy and unnecessary identifiers"
    ]
})

display(excluded_fields)

print("Final feature count:", len(FEATS))
print("Identifiers in feature vector:",
      set(FEATS) & {GROUP, "content_hash_id"})

,field,reason
0,clicks_late,Future label window
1,label,Target variable
2,client_hash_id,Identifier; grouping only
3,content_hash_id,Identifier; joining only
4,Existing product scores / flags,Decision-derived information
5,URLs / domains / private queries,Privacy and unnecessary identifiers


Final feature count: 5
Identifiers in feature vector: set()



## Self-check

- [x] Feature vector and feature notes are documented.
- [x] Missing numeric values are handled with median imputation.
- [x] No categorical columns are selected.
- [x] Future-window and target-derived fields are excluded.
- [x] IDs are used only for grouping and joining.
- [x] Search-volume timestamp remains an open limitation.
- [ ] Run all cells and save the outputs before submission.
